# 16 — Inter-annotator agreement

Assesses coding reliability on a 350-comment reliability sample (250 simple-random + 100 rare-positive enriched, seed 42), independently re-coded by a second annotator blind to the original labels. Agreement is computed between the original labels (from `labelled_comments_final.csv`, "time 1") and the blind re-coding (`inter_annotator_agreement_blind_350.csv`, "time 2"), using raw agreement, Cohen's κ, and positive/negative specific agreement, on the 341 comments whose text matched exactly (9 were excluded for truncation).

Inputs (`data/processed/`): `labelled_comments_final.csv`, `inter_annotator_agreement_blind_350.csv`, `inter_annotator_agreement_key_DO_NOT_OPEN.csv` (the key flags the original-250 subset). Runs from the included data.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Handles VS Code starting the notebook inside /notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

repeat = pd.read_csv(
    DATA_DIR / "inter_annotator_agreement_blind_350.csv",
    encoding="utf-8-sig"
)

original = pd.read_csv(
    DATA_DIR / "labelled_comments_final.csv",
    encoding="utf-8-sig"
)

print("Repeated coding:", repeat.shape)
print("Original dataset:", original.shape)

print("\nRepeated-coding columns:")
print(repeat.columns.tolist())

print("\nMissing repeated labels:")
print(
    repeat[
        [
            "label_1A_annotator2",
            "label_1B_annotator2",
            "label_2_annotator2",
        ]
    ].isna().sum()
)

In [ ]:
assert len(repeat) == 350, "The repeat sample does not contain 350 rows."
assert repeat["agreement_id"].nunique() == 350
assert repeat["comment_id"].nunique() == 350

print("All 350 repeated annotations loaded successfully.")

In [ ]:
import numpy as np
import unicodedata

# The key identifies which rows belonged to the original 250
key = pd.read_csv(
    DATA_DIR / "inter_annotator_agreement_key_DO_NOT_OPEN.csv",
    encoding="utf-8-sig"
)

# Rename the repeated judgements as Time 2
repeat_time2 = repeat.rename(columns={
    "text_clean": "text_clean_time2",
    "label_1A_annotator2": "label_1A_time2",
    "label_1B_annotator2": "label_1B_time2",
    "label_2_annotator2": "label_2_time2",
})

# Select and rename the original judgements as Time 1
original_time1 = original[
    ["comment_id", "text_clean", "label_1A", "label_1B", "label_2"]
].rename(columns={
    "text_clean": "text_clean_time1",
    "label_1A": "label_1A_time1",
    "label_1B": "label_1B_time1",
    "label_2": "label_2_time1",
})

assert original_time1["comment_id"].is_unique

# Match each repeated judgement to its original judgement
iaa = repeat_time2.merge(
    original_time1,
    on="comment_id",
    how="left",
    validate="one_to_one"
)

# Identify the original and enriched portions without relying on row order
original_250_ids = set(key["comment_id"])

iaa["iaa_sample"] = np.where(
    iaa["comment_id"].isin(original_250_ids),
    "original_250",
    "enriched_100"
)

# Check whether exactly the same text was presented both times
def normalise_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    return " ".join(value.split())

time1_text = iaa["text_clean_time1"].map(normalise_text)
time2_text = iaa["text_clean_time2"].map(normalise_text)

iaa["text_integrity"] = [
    "exact" if original_text == repeated_text
    else "time2_truncated" if original_text.startswith(repeated_text)
    else "different"
    for original_text, repeated_text in zip(time1_text, time2_text)
]

iaa["comparable_flag"] = (iaa["text_integrity"] == "exact").astype(int)

# Final checks
time1_labels = [
    "label_1A_time1",
    "label_1B_time1",
    "label_2_time1",
]

assert len(iaa) == 350
assert iaa[time1_labels].notna().all().all()
assert iaa["comment_id"].nunique() == 350

print("Merged rows:", len(iaa))
print("\nSample composition:")
print(iaa["iaa_sample"].value_counts())

print("\nText integrity:")
print(iaa["text_integrity"].value_counts())

In [ ]:
# Inspect 2 different rows

from difflib import SequenceMatcher

different_rows = iaa[
    iaa["text_integrity"] == "different"
].copy()

different_rows["time1_length"] = (
    different_rows["text_clean_time1"].astype(str).str.len()
)

different_rows["time2_length"] = (
    different_rows["text_clean_time2"].astype(str).str.len()
)

different_rows["similarity"] = [
    SequenceMatcher(None, str(time1), str(time2)).ratio()
    for time1, time2 in zip(
        different_rows["text_clean_time1"],
        different_rows["text_clean_time2"]
    )
]

for _, row in different_rows.iterrows():
    print("=" * 80)
    print("Agreement ID:", row["agreement_id"])
    print("Comment ID:", row["comment_id"])
    print("Sample:", row["iaa_sample"])
    print("Similarity:", round(row["similarity"], 3))
    print("\nTIME 1:")
    print(row["text_clean_time1"])
    print("\nTIME 2:")
    print(row["text_clean_time2"])
    print()

In [ ]:
# Correct classification for this
minor_format_ids = ["IAA0024", "IAA0200"]

iaa.loc[
    iaa["agreement_id"].isin(minor_format_ids),
    "text_integrity"
] = "minor_format_difference"

iaa["comparable_flag"] = iaa["text_integrity"].isin(
    ["exact", "minor_format_difference"]
).astype(int)

print(iaa["text_integrity"].value_counts())
print("\nComparable rows:", iaa["comparable_flag"].sum())

# Keep only the directly comparable rows, then remove the temporary fields
iaa = iaa[iaa["comparable_flag"] == 1].copy()
iaa = iaa.drop(
    columns=["iaa_sample", "comparable_flag"],
    errors="ignore"
)

assert len(iaa) == 341

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Remove the temporary sample-splitting fields
iaa = iaa.drop(
    columns=["iaa_sample", "comparable_flag"],
    errors="ignore"
)

assert len(iaa) == 341

LABEL_PAIRS = {
    "1A": ("label_1A_time1", "label_1A_time2"),
    "1B": ("label_1B_time1", "label_1B_time2"),
    "2":  ("label_2_time1", "label_2_time2"),
}

# Ensure labels are numeric
for time1_col, time2_col in LABEL_PAIRS.values():
    iaa[time1_col] = pd.to_numeric(
        iaa[time1_col], errors="raise"
    ).astype(int)

    iaa[time2_col] = pd.to_numeric(
        iaa[time2_col], errors="raise"
    ).astype(int)


results = []

for label, (time1_col, time2_col) in LABEL_PAIRS.items():

    time1 = iaa[time1_col]
    time2 = iaa[time2_col]

    tn, fp, fn, tp = confusion_matrix(
        time1,
        time2,
        labels=[0, 1]
    ).ravel()

    positive_denominator = (2 * tp) + fp + fn
    negative_denominator = (2 * tn) + fp + fn

    results.append({
        "label": label,
        "n": len(iaa),
        "time1_positive_n": int(time1.sum()),
        "time2_positive_n": int(time2.sum()),
        "time1_positive_rate": time1.mean(),
        "time2_positive_rate": time2.mean(),
        "raw_agreement": (time1 == time2).mean(),
        "cohens_kappa": cohen_kappa_score(time1, time2),
        "positive_agreement": (
            2 * tp / positive_denominator
            if positive_denominator else np.nan
        ),
        "negative_agreement": (
            2 * tn / negative_denominator
            if negative_denominator else np.nan
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    })

metrics = pd.DataFrame(results)


# Agreement across all three labels simultaneously
time1_columns = [
    "label_1A_time1",
    "label_1B_time1",
    "label_2_time1",
]

time2_columns = [
    "label_1A_time2",
    "label_1B_time2",
    "label_2_time2",
]

iaa["exact_pattern_agreement"] = (
    iaa[time1_columns].to_numpy()
    == iaa[time2_columns].to_numpy()
).all(axis=1)

exact_pattern_agreement = iaa["exact_pattern_agreement"].mean()

print(metrics.round(3))


print(
    f"\nExact three-label agreement: "
    f"{exact_pattern_agreement:.1%} (n={len(iaa)})"
)
